# 03 - Modelagem de Risco

Notebook de modelagem sobre `data/gold/base_modelagem_risco.csv` com:

- baseline: `LogisticRegression`
- modelo principal: `RandomForestClassifier`
- métricas: **ROC-AUC, Recall, F1, Matriz de Confusão**
- interpretação: **importância de variáveis**


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, recall_score, f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

gold_path = PROJECT_ROOT / "data" / "gold" / "base_modelagem_risco.csv"
df = pd.read_csv(gold_path, low_memory=False)
print("Base gold:", df.shape)
print(df["target_risco"].value_counts(normalize=True).rename("proporcao").round(3))



In [ ]:
# Evita vazamento temporal: remover colunas t+1 / deltas futuros do conjunto de features
leakage_cols = [
    "ida_next",
    "ian_next",
    "defasagem_next",
    "delta_ida_next",
    "delta_ian_next",
    "delta_defasagem_next",
]

feature_cols = [
    "ano_referencia",
    "idade",
    "ano_ingresso",
    "inde",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "defasagem",
    "matematica",
    "portugues",
]

feature_cols = [c for c in feature_cols if c in df.columns and c not in leakage_cols]
X = df[feature_cols].copy()
y = df["target_risco"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Features usadas:", feature_cols)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)


In [ ]:
logit_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1200, class_weight="balanced", random_state=42)),
    ]
)

rf_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=1,
        )),
    ]
)

logit_pipe.fit(X_train, y_train)
rf_pipe.fit(X_train, y_train)

proba_logit = logit_pipe.predict_proba(X_test)[:, 1]
pred_logit = (proba_logit >= 0.5).astype(int)

proba_rf = rf_pipe.predict_proba(X_test)[:, 1]
pred_rf = (proba_rf >= 0.5).astype(int)

def get_metrics(y_true, y_pred, y_proba):
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
    }

metrics_logit = get_metrics(y_test, pred_logit, proba_logit)
metrics_rf = get_metrics(y_test, pred_rf, proba_rf)

metrics_df = pd.DataFrame([
    {"modelo": "LogisticRegression (baseline)", **metrics_logit},
    {"modelo": "RandomForest (principal)", **metrics_rf},
]).set_index("modelo")

display(metrics_df.round(4))




In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_logit = confusion_matrix(y_test, pred_logit)
cm_rf = confusion_matrix(y_test, pred_rf)

ConfusionMatrixDisplay(confusion_matrix=cm_logit).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Matriz de Confusão - LogisticRegression")

ConfusionMatrixDisplay(confusion_matrix=cm_rf).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Matriz de Confusão - RandomForest")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
RocCurveDisplay.from_predictions(y_test, proba_logit, name="LogisticRegression")
RocCurveDisplay.from_predictions(y_test, proba_rf, name="RandomForest")
plt.title("Curvas ROC - Baseline vs Modelo Principal")
plt.show()


In [ ]:
# Importância de variáveis (modelo principal)
rf_model = rf_pipe.named_steps["model"]
importances = pd.DataFrame(
    {
        "feature": feature_cols,
        "importancia": rf_model.feature_importances_,
    }
).sort_values("importancia", ascending=False)

display(importances)

plt.figure(figsize=(9, 5))
sns.barplot(data=importances.head(12), x="importancia", y="feature", palette="mako")
plt.title("Top variáveis mais relevantes - RandomForest")
plt.xlabel("Importância")
plt.ylabel("Feature")
plt.show()


In [ ]:
# Score final para inspeção rápida
rf_full = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=1,
        )),
    ]
)
rf_full.fit(X, y)

df_scored = df.copy()
df_scored["score_risco"] = rf_full.predict_proba(X)[:, 1]

scored_path = PROJECT_ROOT / "data" / "gold" / "base_modelagem_risco_scored.csv"
df_scored.to_csv(scored_path, index=False)

print("Arquivo com score salvo em:", scored_path)
display(df_scored[["ra", "ano_referencia", "target_risco", "score_risco"]].head(10))

